In [ ]:
!pip install -q datasets[audio] transformers accelerate evaluate jiwer tensorboard kagglehub

In [ ]:
import kagglehub
import os

dataset_path = kagglehub.dataset_download("hecrws/common-voice-mozilla-dataset")
print(f"Dataset downloaded to: {dataset_path}")

for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        sub_indent = " " * 2 * (level + 1)
        for f in files[:10]:
            print(f"{sub_indent}{f}")
        if len(files) > 10:
            print(f"{sub_indent}... and {len(files) - 10} more files")

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
from pathlib import Path

base = Path(dataset_path)

clips_dir = None
tsv_dir = None
for p in base.rglob("clips"):
    if p.is_dir():
        clips_dir = p
        tsv_dir = p.parent
        break

if clips_dir is None:
    tsv_files = list(base.rglob("train.tsv"))
    if tsv_files:
        tsv_dir = tsv_files[0].parent
        clips_dir = tsv_dir / "clips"

print(f"TSV dir:   {tsv_dir}")
print(f"Clips dir: {clips_dir}")

def load_split(split_name):
    tsv_path = tsv_dir / f"{split_name}.tsv"
    if not tsv_path.exists():
        print(f"  {split_name}.tsv not found, skipping")
        return None
    df = pd.read_csv(tsv_path, sep="\t")
    print(f"  {split_name}: {len(df)} rows")
    df["audio_path"] = df["path"].apply(lambda p: str(clips_dir / p))
    df = df[df["audio_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)
    print(f"  {split_name} (after filtering missing audio): {len(df)} rows")
    ds = Dataset.from_pandas(df[["audio_path", "sentence"]])
    return ds

print("Loading splits...")
train_ds = load_split("train")
dev_ds = load_split("dev")
test_ds = load_split("test")

if train_ds is None:
    print("No train.tsv found, trying validated.tsv...")
    validated = load_split("validated")
    if validated is not None:
        split = validated.train_test_split(test_size=0.1, seed=42)
        train_ds = split["train"]
        test_ds = split["test"]

if dev_ds is not None and train_ds is not None:
    from datasets import concatenate_datasets
    train_ds = concatenate_datasets([train_ds, dev_ds])
    print(f"  Combined train+dev: {len(train_ds)} rows")

common_voice = DatasetDict({
    "train": train_ds,
    "test": test_ds,
})

print(f"\nFinal dataset:")
print(f"  Train: {len(common_voice['train'])} samples")
print(f"  Test:  {len(common_voice['test'])} samples")

In [ ]:
import torch
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
)

MODEL_ID = "openai/whisper-large-v3-turbo"

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID, language="Arabic", task="transcribe")
processor = WhisperProcessor.from_pretrained(MODEL_ID, language="Arabic", task="transcribe")

# turbo's checkpoint on HF is fp16. Recent transformers respects that and
# loads the model in fp16, which is incompatible with fp32/bf16 training
# (causes "bias type c10::Half" mismatch in the encoder's Conv1d). Loading
# explicitly in fp32 upcasts the weights once at load time and gives us a
# normal fp32 model that trains cleanly.
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float32)

model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

print(f"Model parameters: {model.num_parameters():,}")
print(f"Model dtype:      {next(model.parameters()).dtype}")

In [ ]:
import librosa

TRAIN_SUBSET = 10000
TEST_SUBSET = 1000

if TRAIN_SUBSET is not None and len(common_voice["train"]) > TRAIN_SUBSET:
    common_voice["train"] = common_voice["train"].shuffle(seed=42).select(range(TRAIN_SUBSET))
if TEST_SUBSET is not None and len(common_voice["test"]) > TEST_SUBSET:
    common_voice["test"] = common_voice["test"].shuffle(seed=42).select(range(TEST_SUBSET))

print(f"Train: {len(common_voice['train'])}")
print(f"Test:  {len(common_voice['test'])}")


def prepare_batch(batch):
    input_features_list = []
    labels_list = []

    for audio_path, sentence in zip(batch["audio_path"], batch["sentence"]):
        try:
            waveform, sr = librosa.load(audio_path, sr=16000, mono=True)
            if waveform is None or len(waveform) == 0:
                continue
            feats = feature_extractor(
                waveform, sampling_rate=16000
            ).input_features[0]
            labels = tokenizer(sentence).input_ids
            input_features_list.append(feats)
            labels_list.append(labels)
        except Exception:
            continue

    return {"input_features": input_features_list, "labels": labels_list}


common_voice["train"].set_transform(prepare_batch)
common_voice["test"].set_transform(prepare_batch)

print("\nLazy transform attached. Audio loaded via librosa per-batch.")

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        # Cast input_features to bf16 to match the model during eval/generate.
        # bf16_full_eval=True puts the model in bf16 for evaluation, and
        # generate() doesn't autocast inputs — so without this cast the
        # encoder's Conv1d sees fp32 inputs vs bf16 weights and errors.
        # Training is unaffected: autocast accepts bf16 inputs to fp32 params.
        batch["input_features"] = batch["input_features"].to(torch.bfloat16)
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [ ]:
import re
import evaluate

wer_metric = evaluate.load("wer")

# Arabic text normalization — single biggest WER win for Arabic ASR.
# Whisper often produces correct words with different orthographic variants
# (alef أ/إ/آ/ا, ya ى/ي, ta marbuta ة/ه, with/without diacritics). Normalizing
# both predictions and references collapses these to a canonical form before
# scoring, dropping WER 10-20 points without changing the model.
_DIACRITICS = re.compile(r"[ً-ٰٟۖ-ۭـ]")
_PUNCT = re.compile(r"[^\w\s؀-ۿ]")

def normalize_arabic(text: str) -> str:
    if not text:
        return ""
    text = _DIACRITICS.sub("", text)              # tashkeel + tatweel
    text = re.sub(r"[إأآا]", "ا", text)            # alef variants
    text = re.sub(r"ى", "ي", text)                 # alef maksura → ya
    text = re.sub(r"ة", "ه", text)                 # ta marbuta → ha
    text = re.sub(r"ؤ", "و", text)                 # waw with hamza
    text = re.sub(r"ئ", "ي", text)                 # ya with hamza
    text = _PUNCT.sub(" ", text)                  # strip punctuation
    text = re.sub(r"\s+", " ", text).strip()
    return text


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_str = [normalize_arabic(s) for s in pred_str]
    label_str = [normalize_arabic(s) for s in label_str]

    # Drop empty references after normalization to avoid divide-by-zero
    pairs = [(p, l) for p, l in zip(pred_str, label_str) if l.strip()]
    if not pairs:
        return {"wer": 100.0}
    pred_str, label_str = zip(*pairs)

    wer = 100 * wer_metric.compute(predictions=list(pred_str), references=list(label_str))
    return {"wer": wer}

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

OUTPUT_DIR = "./whisper-turbo-arabic"

model.config.use_cache = False
model.generation_config.use_cache = False

# Now that the model loads in fp32 (cell 3), bf16 mixed-precision training
# works cleanly: fp32 params + bf16 autocast forward, and bf16_full_eval
# casts to bf16 for evaluation/generate so dtypes stay consistent.
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_steps=200,
    max_steps=2000,
    gradient_checkpointing=True,
    bf16=True,
    bf16_full_eval=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    eval_steps=500,
    save_steps=500,
    logging_steps=25,
    predict_with_generate=True,
    report_to="tensorboard",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print("Starting training...")
trainer.train()

In [ ]:
eval_results = trainer.evaluate()
print(f"\nFinal WER: {eval_results['eval_wer']:.2f}%")

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/final")
processor.save_pretrained(f"{OUTPUT_DIR}/final")
print(f"Model saved to {OUTPUT_DIR}/final")

from google.colab import drive
drive.mount("/content/drive")

!cp -r {OUTPUT_DIR}/final /content/drive/MyDrive/whisper-turbo-arabic
print("Model copied to Google Drive!")

In [ ]:
from transformers import pipeline
import librosa

pipe = pipeline(
    "automatic-speech-recognition",
    model=f"{OUTPUT_DIR}/final",
    device=0,
)

test_tsv = tsv_dir / "test.tsv"
if not test_tsv.exists():
    test_tsv = tsv_dir / "validated.tsv"

test_df = pd.read_csv(test_tsv, sep="\t", nrows=5)
sample_path = str(clips_dir / test_df.iloc[0]["path"])
sample_sentence = test_df.iloc[0]["sentence"]

result = pipe(sample_path, generate_kwargs={"language": "arabic", "task": "transcribe"})

print(f"Reference:  {sample_sentence}")
print(f"Predicted:  {result['text']}")